# Ingestion with LlamaParse: a prototype

A trial of the new ingestion steps on the seven pages parsed in notebook 04. It reads that cached parse,
so no LlamaParse credits are spent.

| Step | What it does |
|---|---|
| 1. Load | Read the cached LlamaParse result: one markdown string per page, and the cropped images |
| 2. Chunk | One chunk per page. A page longer than `CHUNK_MAX_CHARACTERS` is split at its headings. A table is never split. |
| 3. Images | Attach each page's cropped figures and charts to that page's chunks. Table crops are skipped. |
| 4. Describe | A chunk with a table or an image gets a searchable description from the vision model, as in `ingest.py` today |
| 5. Store | Embed and save in a test Chroma store with the page number on every chunk |
| 6. Search | Ask questions about these pages and check that the right page comes back first |

Images are stored as file paths, not base64, so a later API can serve them as links.

In [1]:
import base64
import json
import sys
import time
from pathlib import Path

from langchain_chroma import Chroma
from langchain_core.documents import Document

PROJECT_ROOT = Path.cwd().parent  # this notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

from multimodal_rag import config
from multimodal_rag.ingest import describe_chunk, description_model
from multimodal_rag.store import NemotronEmbeddings

LLAMAPARSE_OUTPUT = PROJECT_ROOT / "notebooks" / "llamaparse_output"
PARSE_RESULT_PATH = LLAMAPARSE_OUTPUT / "sample_result.json"   # written by notebook 04
IMAGE_DIRECTORY = LLAMAPARSE_OUTPUT / "images"                 # written by notebook 04
DESCRIPTIONS_PATH = LLAMAPARSE_OUTPUT / "descriptions.json"    # cache, so a re-run skips the model calls
TEST_STORE_DIRECTORY = LLAMAPARSE_OUTPUT / "chroma_test"

/home/aikosh/multimodal-rag-pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the parse result

In [2]:
parse_result = json.loads(PARSE_RESULT_PATH.read_text())

markdown_by_page = {}
for page in parse_result["markdown"]["pages"]:
    markdown_by_page[page["page_number"]] = page["markdown"]

for page_number, markdown in markdown_by_page.items():
    print(f"page {page_number:3}: {len(markdown):5} characters")

page   4:   795 characters
page   8:  1268 characters
page  13:  4099 characters
page  26:  5055 characters
page 128:  3542 characters
page 139:  2024 characters
page 142:  3971 characters


## 2. Chunk: one chunk per page, split at headings when too long

`split_into_sections` cuts a page's markdown at every heading line (a line starting with `#`).
Lines inside a fenced code block, such as the Mermaid diagram on page 8, are never treated as headings.

`group_sections` then joins neighbouring sections back together, as long as the result stays within
`CHUNK_MAX_CHARACTERS`. A short page stays one chunk. A section that is longer than the limit on its own,
usually a big table, becomes a chunk by itself instead of being cut. A chunk shorter than
`CHUNK_COMBINE_UNDER_CHARACTERS` always takes the next section too, so a heading is never left on its own.

In [3]:
def split_into_sections(markdown):
    """Cut a page's markdown at its headings. Each section starts with its heading."""
    sections = []
    current_lines = []
    inside_code_block = False

    for line in markdown.splitlines():
        if line.startswith("```"):
            inside_code_block = not inside_code_block
        is_heading = line.startswith("#") and not inside_code_block

        if is_heading and current_lines:
            sections.append("\n".join(current_lines).strip())
            current_lines = []
        current_lines.append(line)

    if current_lines:
        sections.append("\n".join(current_lines).strip())
    return [section for section in sections if section]


def group_sections(sections, max_characters, min_characters):
    """Join neighbouring sections into chunks of at most max_characters.

    A section longer than max_characters stays whole. A chunk shorter than min_characters,
    such as a lone heading, always takes the next section too.
    """
    chunks = []
    current = ""
    for section in sections:
        too_long_together = len(current) + len(section) + 2 > max_characters
        if current and too_long_together and len(current) >= min_characters:
            chunks.append(current)
            current = ""
        current = f"{current}\n\n{section}" if current else section
    if current:
        chunks.append(current)
    return chunks

## 3. Attach images

LlamaParse names each crop after its page and kind: `page_8_chart_1_v2.jpg`, `page_4_image_1_v2.jpg`,
`page_139_table_1_v2.jpg`. Figures and charts are attached to every chunk of their page. Table crops are left out,
because the table HTML in the chunk already holds the data and the crop would only add tokens.

In [4]:
def figure_images_on_page(page_number):
    """Paths of the cropped figures and charts on a page. Table crops are skipped."""
    crops = sorted(IMAGE_DIRECTORY.glob(f"page_{page_number}_*"))
    return [path for path in crops if "_table_" not in path.name]


def build_chunks(markdown_by_page):
    """Turn every page into one or more chunks, each with its page number and images."""
    chunks = []
    for page_number, markdown in markdown_by_page.items():
        sections = split_into_sections(markdown)
        page_chunks = group_sections(sections, config.CHUNK_MAX_CHARACTERS, config.CHUNK_COMBINE_UNDER_CHARACTERS)
        images = figure_images_on_page(page_number)

        for number, text in enumerate(page_chunks, start=1):
            chunks.append({
                "id": f"page{page_number}-{number}",
                "page_number": page_number,
                "text": text,
                "image_paths": [str(path.relative_to(PROJECT_ROOT)) for path in images],
            })
    return chunks


chunks = build_chunks(markdown_by_page)
print(f"{len(chunks)} chunks from {len(markdown_by_page)} pages\n")
for chunk in chunks:
    tables = chunk["text"].count("<table")
    print(f"{chunk['id']:<10} {len(chunk['text']):5} characters, {tables} table(s), {len(chunk['image_paths'])} image(s)")

11 chunks from 7 pages

page4-1      795 characters, 0 table(s), 2 image(s)
page8-1     1268 characters, 0 table(s), 1 image(s)
page13-1    2931 characters, 0 table(s), 0 image(s)
page13-2    1166 characters, 0 table(s), 0 image(s)
page26-1    1679 characters, 1 table(s), 0 image(s)
page26-2    1352 characters, 0 table(s), 0 image(s)
page26-3    2020 characters, 1 table(s), 0 image(s)
page128-1   1951 characters, 2 table(s), 0 image(s)
page128-2   1589 characters, 1 table(s), 0 image(s)
page139-1   2026 characters, 1 table(s), 0 image(s)
page142-1   3972 characters, 1 table(s), 0 image(s)


The chunks in full, to check where the pages were split:

In [5]:
for chunk in chunks:
    print(f"==================== {chunk['id']}  images: {chunk['image_paths']}")
    print(chunk["text"])
    print()

==================== page4-1  images: ['notebooks/llamaparse_output/images/page_4_image_1_v2.jpg', 'notebooks/llamaparse_output/images/page_4_image_2_v2.jpg']
## **AI Is Driving the Largest Infrastructure Buildout in Human History**

**Data centers are becoming AI factories.**

Land, power, cooling, networking, and systems must scale in parallel.
Every company will use AI. Every country will build AI infrastructure.
This is the next industrial revolution. We are only a few hundred billion dollars into it. Trillions more still need to be built.

image: infographic showing the evolution of DSX technology and its integration into data center infrastructure

**Reference Designs, Best Practices, Methodologies**

**Rubin DSX Reference Designs**

**DSX Ecosystem**

**Omniverse DSX Blueprint**

**DSX Max-Q**

**DSX Flex**

**DSX Exchange**

**DSX Sim**

**DSX Libraries, APIs, Software**

**Chips, Systems, Facilities**

**Power**

**Liquid Cooling**

==================== page8-1  images: ['note

## 4. Describe the chunks with a table or an image

Same rule as `ingest.py` today: the description is what gets embedded, so the chunk can be found by meaning
rather than by raw HTML. The vision model receives the chunk text (with its table HTML) and the cropped images.
`describe_chunk` comes from `ingest.py`. The table HTML is already inside the text, so its `tables` argument is empty.

Descriptions are cached in `descriptions.json`, so running this notebook again does not call the model again.

In [6]:
def needs_description(chunk):
    return "<table" in chunk["text"] or bool(chunk["image_paths"])


def image_as_base64(path):
    return base64.b64encode((PROJECT_ROOT / path).read_bytes()).decode()


descriptions = json.loads(DESCRIPTIONS_PATH.read_text()) if DESCRIPTIONS_PATH.exists() else {}
llm = description_model()

for chunk in chunks:
    if not needs_description(chunk) or chunk["id"] in descriptions:
        continue
    start = time.time()
    images = [image_as_base64(path) for path in chunk["image_paths"]]
    descriptions[chunk["id"]] = describe_chunk(llm, chunk["text"], [], images)
    DESCRIPTIONS_PATH.write_text(json.dumps(descriptions, indent=1))
    print(f"described {chunk['id']} in {time.time() - start:.0f}s")

described = [chunk["id"] for chunk in chunks if chunk["id"] in descriptions]
print(f"{len(described)} of {len(chunks)} chunks have a description: {described}")

described page142-1 in 40s
8 of 11 chunks have a description: ['page4-1', 'page8-1', 'page26-1', 'page26-3', 'page128-1', 'page128-2', 'page139-1', 'page142-1']


The two diagram descriptions, since the page markdown says little about them:

In [7]:
for chunk_id in ["page4-1", "page8-1"]:
    print(f"==================== {chunk_id}")
    print(descriptions.get(chunk_id, "(no description)"))
    print()

==================== page4-1
SEARCHABLE DESCRIPTION:

**Content Overview and Key Facts**
This content describes the massive scale of AI infrastructure development, characterizing it as the "Largest Infrastructure Buildout in Human History." It posits that data centers are evolving into "AI factories." Key financial data points indicate that while a "few hundred billion dollars" have already been invested, "Trillions more still need to be built." The scope is global and universal, stating that "Every company will use AI" and "Every country will build AI infrastructure." The text identifies this shift as the "next industrial revolution."

**Main Topics and Concepts**
The primary subject is AI-driven data center infrastructure. Core concepts include the parallel scaling of land, power, cooling, networking, and systems. The content outlines a comprehensive ecosystem involving:
*   **Reference Designs & Methodologies:** Rubin DSX Reference Designs, DSX Ecosystem, Omniverse DSX Blueprint.
* 

## 5. Store in a test Chroma store

Each chunk becomes one document:

- **Embedded text:** the description when there is one, otherwise the chunk text.
- **Metadata:** `page_number`, and `original_content` with the chunk text and its image paths, for the answering model.

The test store lives in `notebooks/llamaparse_output/chroma_test`, apart from the pipeline's store.

In [8]:
def build_documents(chunks, descriptions):
    documents = []
    for chunk in chunks:
        searchable_text = descriptions.get(chunk["id"], chunk["text"])
        original_content = {"text": chunk["text"], "image_paths": chunk["image_paths"]}
        metadata = {"page_number": chunk["page_number"], "original_content": json.dumps(original_content)}
        documents.append(Document(page_content=searchable_text, metadata=metadata, id=chunk["id"]))
    return documents


test_store = Chroma(
    persist_directory=str(TEST_STORE_DIRECTORY),
    embedding_function=NemotronEmbeddings(),
    collection_metadata={"hnsw:space": "cosine"},
)
test_store.reset_collection()
test_store.add_documents(build_documents(chunks, descriptions))
print(f"Stored {len(chunks)} chunks in {TEST_STORE_DIRECTORY.relative_to(PROJECT_ROOT)}")

Stored 11 chunks in notebooks/llamaparse_output/chroma_test


## 6. Search

Each question names the page that holds its answer. The last one is not answered anywhere in these pages,
so all its scores should be low, ideally under `MIN_RELEVANCE_SCORE`.

In [9]:
QUESTIONS = [
    ("What was NVIDIA's total revenue in fiscal year 2024?", 139),
    ("Which director nominees are not independent?", 26),
    ("What does NemoClaw connect to in the agentic AI architecture?", 8),
    ("What share of revenue went to research and development in fiscal 2026?", 128),
    ("How much did NVIDIA spend repurchasing shares in fiscal 2026?", 142),
    ("What are the DSX reference designs for AI factories?", 4),
    ("How many employees does NVIDIA have?", None),
]

first_place = 0
for question, expected_page in QUESTIONS:
    results = test_store.similarity_search_with_relevance_scores(question, k=3)
    top_page = results[0][0].metadata["page_number"]
    verdict = "no answer expected" if expected_page is None else ("OK" if top_page == expected_page else "MISS")
    if top_page == expected_page:
        first_place += 1

    print(f"{question}\n   expected page {expected_page}, got page {top_page}: {verdict}")
    for document, score in results:
        preview = " ".join(document.page_content.split())[:90]
        print(f"   {score:.3f}  page {document.metadata['page_number']:3}  {preview}")
    print()

answerable = sum(1 for _, page in QUESTIONS if page is not None)
print(f"Right page first for {first_place} of {answerable} answerable questions")

What was NVIDIA's total revenue in fiscal year 2024?
   expected page 139, got page 139: OK
   0.587  page 139  # SEARCHABLE DESCRIPTION: NVIDIA Corporation Consolidated Statements of Income ## 1. Docum
   0.468  page 142  **SEARCHABLE DESCRIPTION: NVIDIA Corporation Consolidated Statements of Shareholders' Equi
   0.443  page 128  **SEARCHABLE DESCRIPTION** --- ### 1. KEY FACTS, NUMBERS, AND DATA POINTS **Reporting Peri

Which director nominees are not independent?
   expected page 26, got page 26: OK
   0.500  page  26  **SEARCHABLE DESCRIPTION: ELECTION OF DIRECTORS (PROPOSAL 1) — BOARD NOMINEES AND GOVERNAN
   0.389  page  26  # SEARCHABLE DESCRIPTION: Director Nominee Skills, Competencies, and Attributes Matrix ## 
   0.348  page  26  ### **Recent Refreshment and Nominee Qualifications** Our director nominees reflect a broa

What does NemoClaw connect to in the agentic AI architecture?
   expected page 8, got page 8: OK
   0.574  page   8  **SEARCHABLE DESCRIPTION** --- ### 1. KEY

## Findings

**The approach works.** The 7 pages became 11 chunks, and the right page came first for all 6 answerable questions, including the diagram questions (pages 4 and 8), the checkmark table (page 26) and the three financial tables.

- **Chunking.** One chunk per page holds for short pages: the spreads, page 139 and page 142. Longer pages split at their headings (13 into 2 chunks, 26 into 3, 128 into 2), and no table was cut.
- **Lone headings, fixed.** The first run left `## NVIDIA Corporation and Subsidiaries` (42 characters) as a chunk on its own, because heading and table together passed `CHUNK_MAX_CHARACTERS`. That empty chunk ranked first for an unrelated question, and the page 142 table ranked only third for the share repurchase question. Now a chunk shorter than `CHUNK_COMBINE_UNDER_CHARACTERS` always takes the next section. Page 142 is one chunk and ranks first (0.56).
- **Descriptions.** 8 of the 11 chunks have a table or an image and were described, in 19 to 50 seconds each with the free Nemotron model. The diagram descriptions are good:
  - page 4 names every DSX module from the crop, which the page markdown barely mentions;
  - page 8 lists every node and connection of the agentic AI diagram.
- **Images.** Stored as paths such as `notebooks/llamaparse_output/images/page_8_chart_1_v2.jpg`. Page 4 carries two overlapping crops, which the description model handled; the smaller one could be dropped later.

**To watch on the full report:**

- **The cutoff.** The question with no answer in these pages still scored 0.38 at the top, above the 0.3 cutoff, so `MIN_RELEVANCE_SCORE` needs re-checking on all 175 pages. (The full report does give an employee count in the 10-K, so that question needs replacing for the full run.)
- **Description time.** Most report pages hold a table, so expect 100 or more description calls: around an hour on the free model, much less on a paid one.
- **Printed page numbers.** These were not requested in this parse. The full run should turn on `extract_printed_page_number` and `merge_continued_tables`.
